In [ ]:
%pip install dbx_test

In [ ]:
dbutils.library.restartPython()

In [ ]:
from dbx_test import NotebookTestFixture, run_notebook_tests
from pyspark.sql.functions import col, row_number, dense_rank, regexp_replace, lit, expr
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime, timedelta

In [ ]:
# ============================================================================
# Helper Functions
# Extracted from pipeline logic for unit-testable transformations.
# ============================================================================

def apply_dlt_expectations(df, no_rescued_data=True, valid_id=True, valid_operation=True):
    """Simulates @dlt.expect_or_drop expectations from python_cdc.ipynb."""
    result = df
    if no_rescued_data and "_rescued_data" in df.columns:
        result = result.filter(col("_rescued_data").isNull())
    if valid_id:
        result = result.filter(col("id").isNotNull())
    if valid_operation:
        result = result.filter(col("operation").isin(["APPEND", "DELETE", "UPDATE"]))
    return result


def deduplicate_cdc_data(df, id_col="id", order_col="operation_date"):
    """Simulates SEQUENCE BY deduplication in dlt.apply_changes."""
    w = Window.partitionBy(id_col).orderBy(col(order_col).desc())
    return df.withColumn("_rank", row_number().over(w)).filter("_rank = 1").drop("_rank")


def apply_changes_scd1(source_df, target_df, keys=None, except_cols=None):
    """Simulates dlt.apply_changes() SCD1 behavior from python_cdc.ipynb."""
    keys = keys or ["id"]
    except_cols = except_cols or ["operation", "operation_date", "_rescued_data"]
    delete_ids = set(r["id"] for r in source_df.filter("operation = 'DELETE'").select(keys).collect())
    upserts = source_df.filter("operation != 'DELETE'")
    upsert_ids = set(r["id"] for r in upserts.select(keys).collect())
    all_affected = delete_ids | upsert_ids
    target_filtered = target_df.filter(~col("id").isin(list(all_affected)))
    existing_except = [c for c in except_cols if c in upserts.columns]
    upserts_prepared = upserts.drop(*existing_except)
    result_cols = target_filtered.columns
    upserts_final = upserts_prepared.select(
        *[col(c) if c in upserts_prepared.columns else lit(None).alias(c) for c in result_cols]
    )
    return target_filtered.union(upserts_final)


def process_cdf_for_gold(cdf_df, id_col="id"):
    """Process CDF data: deduplicate by commit version, filter out preimages."""
    w = Window.partitionBy(id_col).orderBy(col("_commit_version").desc())
    return (cdf_df
            .withColumn("_rank", dense_rank().over(w))
            .filter("_rank = 1 AND _change_type != 'update_preimage'")
            .drop("_commit_version", "_rank"))


def clean_address(df, address_col="address"):
    """Clean address field by removing embedded quotes."""
    return df.withColumn(address_col, regexp_replace(col(address_col), '"', ""))


def get_table_names_from_folders(folder_listing):
    """Extract table names from dbutils.fs.ls() folder listing."""
    return [f.name[:-1] for f in folder_listing]

In [ ]:
# Common schema and timestamp used across test classes
TS = datetime(2024, 1, 1, 10, 0, 0)

CDC_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("firstname", StringType(), True),
    StructField("address", StringType(), True),
    StructField("email", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("operation_date", TimestampType(), True),
    StructField("_rescued_data", StringType(), True),
])

SILVER_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("firstname", StringType(), True),
    StructField("address", StringType(), True),
    StructField("email", StringType(), True),
])

CDF_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("firstname", StringType(), True),
    StructField("_change_type", StringType(), True),
    StructField("_commit_version", LongType(), True),
])

In [ ]:
class TestDLTExpectations(NotebookTestFixture):
    """Tests for @dlt.expect_or_drop data quality rules."""

    def test_valid_records_pass(self):
        df = spark.createDataFrame(
            [(1, "Alice", "123 Main", "a@e.com", "APPEND", TS, None)], CDC_SCHEMA)
        assert apply_dlt_expectations(df).count() == 1

    def test_null_id_dropped(self):
        df = spark.createDataFrame(
            [(None, "Ghost", "Addr", "g@e.com", "APPEND", TS, None)], CDC_SCHEMA)
        assert apply_dlt_expectations(df).count() == 0

    def test_invalid_operation_dropped(self):
        for op in ["INVALID", "MERGE", "UPSERT", ""]:
            df = spark.createDataFrame(
                [(1, "A", "Addr", "a@e.com", op, TS, None)], CDC_SCHEMA)
            assert apply_dlt_expectations(df).count() == 0, f"Operation '{op}' should be dropped"

    def test_valid_operations_pass(self):
        for op in ["APPEND", "UPDATE", "DELETE"]:
            df = spark.createDataFrame(
                [(1, "A", "Addr", "a@e.com", op, TS, None)], CDC_SCHEMA)
            assert apply_dlt_expectations(df).count() == 1, f"Operation '{op}' should pass"

    def test_rescued_data_dropped(self):
        df = spark.createDataFrame(
            [(1, "A", "Addr", "a@e.com", "APPEND", TS, '{"x":1}')], CDC_SCHEMA)
        assert apply_dlt_expectations(df).count() == 0

    def test_mixed_quality_filters_correctly(self):
        df = spark.createDataFrame([
            (1, "Good", "A", "g@e.com", "APPEND", TS, None),
            (None, "NullID", "A", "n@e.com", "APPEND", TS, None),
            (3, "BadOp", "A", "b@e.com", "INVALID", TS, None),
            (4, "Rescued", "A", "r@e.com", "UPDATE", TS, '{"extra":1}'),
            (5, "AlsoGood", "A", "a@e.com", "DELETE", TS, None),
        ], CDC_SCHEMA)
        assert apply_dlt_expectations(df).count() == 2

In [ ]:
class TestDeduplication(NotebookTestFixture):
    """Tests for CDC deduplication (SEQUENCE BY operation_date)."""

    def test_keeps_single_record_per_id(self):
        for num_dupes in [1, 3, 5]:
            data = [(1, f"V{i}", f"A{i}", f"e{i}@t.com", "UPDATE",
                     TS + timedelta(hours=i), None) for i in range(num_dupes)]
            result = deduplicate_cdc_data(spark.createDataFrame(data, CDC_SCHEMA))
            assert result.count() == 1, f"Expected 1 row for {num_dupes} dupes, got {result.count()}"

    def test_keeps_latest_record(self):
        data = [
            (1, "Old", "OldA", "old@e.com", "APPEND", TS, None),
            (1, "Mid", "MidA", "mid@e.com", "UPDATE", TS + timedelta(hours=1), None),
            (1, "New", "NewA", "new@e.com", "UPDATE", TS + timedelta(hours=2), None),
        ]
        row = deduplicate_cdc_data(spark.createDataFrame(data, CDC_SCHEMA)).collect()[0]
        assert row["firstname"] == "New"

    def test_deduplication_across_multiple_ids(self):
        data = [(id_val, f"N{i}", f"A{i}", f"e{i}@t.com", "UPDATE",
                 TS + timedelta(hours=i), None)
                for i, id_val in enumerate([1, 1, 2, 2, 3])]
        result = deduplicate_cdc_data(spark.createDataFrame(data, CDC_SCHEMA))
        assert sorted([r["id"] for r in result.collect()]) == [1, 2, 3]

    def test_empty_dataframe(self):
        assert deduplicate_cdc_data(spark.createDataFrame([], CDC_SCHEMA)).count() == 0

In [ ]:
class TestApplyChangesSCD1(NotebookTestFixture):
    """Tests for dlt.apply_changes() SCD1 merge logic."""

    def run_setup(self):
        self.target = spark.createDataFrame([
            (1, "Alice", "123 Main St", "alice@e.com"),
            (2, "Bob", "456 Oak Ave", "bob@e.com"),
        ], SILVER_SCHEMA)

    def test_append_inserts_new(self):
        src = spark.createDataFrame(
            [(3, "Charlie", "789 Pine", "c@e.com", "APPEND", TS, None)], CDC_SCHEMA)
        result = apply_changes_scd1(src, self.target)
        assert result.count() == 3

    def test_update_modifies_existing(self):
        src = spark.createDataFrame(
            [(1, "Alice V2", "999 New", "a2@e.com", "UPDATE", TS, None)], CDC_SCHEMA)
        row = apply_changes_scd1(src, self.target).filter(col("id") == 1).collect()[0]
        assert row["firstname"] == "Alice V2"

    def test_delete_removes_record(self):
        src = spark.createDataFrame(
            [(1, None, None, None, "DELETE", TS, None)], CDC_SCHEMA)
        result = apply_changes_scd1(src, self.target)
        assert result.count() == 1
        assert result.filter(col("id") == 1).count() == 0

    def test_mixed_operations(self):
        src = spark.createDataFrame([
            (1, "Alice V2", "New Addr", "a2@e.com", "UPDATE", TS, None),
            (2, None, None, None, "DELETE", TS, None),
            (3, "Charlie", "789 Pine", "c@e.com", "APPEND", TS, None),
        ], CDC_SCHEMA)
        result = apply_changes_scd1(src, self.target)
        assert result.count() == 2
        assert result.filter(col("id") == 2).count() == 0
        assert result.filter(col("id") == 3).count() == 1

    def run_cleanup(self):
        self.target = None

In [ ]:
class TestCDFProcessing(NotebookTestFixture):
    """Tests for Change Data Feed processing."""

    def test_filters_preimage(self):
        df = spark.createDataFrame(
            [(1, "Before", "update_preimage", 2), (1, "After", "update_postimage", 2)],
            CDF_SCHEMA)
        result = process_cdf_for_gold(df)
        assert result.count() == 1
        assert result.collect()[0]["firstname"] == "After"

    def test_keeps_latest_version(self):
        df = spark.createDataFrame([
            (1, "V1", "insert", 1),
            (1, "V2", "update_postimage", 2),
            (1, "V3", "update_postimage", 3),
        ], CDF_SCHEMA)
        result = process_cdf_for_gold(df)
        assert result.count() == 1
        assert result.collect()[0]["firstname"] == "V3"

    def test_change_type_selection(self):
        for types, expected in [(["insert"], "insert"), (["delete"], "delete")]:
            data = [(1, f"N_{t}", t, i + 1) for i, t in enumerate(types)]
            result = process_cdf_for_gold(spark.createDataFrame(data, CDF_SCHEMA))
            assert result.collect()[0]["_change_type"] == expected

In [ ]:
class TestDataCleaning(NotebookTestFixture):
    """Tests for data cleaning transformations."""

    def test_address_cleaning(self):
        schema = StructType([
            StructField("id", LongType(), True),
            StructField("address", StringType(), True),
        ])
        cases = [
            ('"123 Main St"', "123 Main St"),
            ("456 Oak Ave", "456 Oak Ave"),
            ("No quotes here", "No quotes here"),
            ('""', ""),
        ]
        for input_addr, expected in cases:
            result = clean_address(spark.createDataFrame([(1, input_addr)], schema))
            assert result.collect()[0]["address"] == expected, f"Failed for '{input_addr}'"

In [ ]:
class TestDynamicPipeline(NotebookTestFixture):
    """Tests for dynamic table discovery and path generation."""

    def test_folder_to_table_names(self):
        class F:
            def __init__(self, n): self.name = n
        folders = [F("customers/"), F("orders/"), F("products/")]
        assert get_table_names_from_folders(folders) == ["customers", "orders", "products"]

    def test_empty_folder_listing(self):
        assert get_table_names_from_folders([]) == []

In [ ]:
class TestEndToEndPipeline(NotebookTestFixture):
    """Integration tests simulating the full CDC pipeline flow."""

    def test_full_cdc_flow(self):
        # Insert 3 records
        raw = spark.createDataFrame([
            (1, "Alice", "123 Main", "a@e.com", "APPEND", TS, None),
            (2, "Bob", "456 Oak", "b@e.com", "APPEND", TS, None),
            (3, "Charlie", "789 Pine", "c@e.com", "APPEND", TS, None),
        ], CDC_SCHEMA)
        target = spark.createDataFrame([], SILVER_SCHEMA)
        result = apply_changes_scd1(apply_dlt_expectations(raw), target)
        assert result.count() == 3

        # Update Alice
        updates = spark.createDataFrame(
            [(1, "Alice V2", "999 New", "a2@e.com", "UPDATE", TS + timedelta(hours=1), None)],
            CDC_SCHEMA)
        result = apply_changes_scd1(apply_dlt_expectations(updates), result)
        assert result.filter(col("id") == 1).collect()[0]["firstname"] == "Alice V2"

        # Delete Bob
        deletes = spark.createDataFrame(
            [(2, None, None, None, "DELETE", TS + timedelta(hours=2), None)], CDC_SCHEMA)
        result = apply_changes_scd1(apply_dlt_expectations(deletes), result)
        assert result.count() == 2
        assert result.filter(col("id") == 2).count() == 0

    def test_large_batch_deduplication(self):
        data = [(i % 100, f"N{i}", f"A{i}", f"e{i}@t.com", "UPDATE",
                 TS + timedelta(seconds=i), None) for i in range(1000)]
        result = deduplicate_cdc_data(spark.createDataFrame(data, CDC_SCHEMA))
        assert result.count() == 100

In [ ]:
import json, inspect

fixtures = [
    v for v in globals().values()
    if inspect.isclass(v) and issubclass(v, NotebookTestFixture) and v is not NotebookTestFixture
]
results = run_notebook_tests(fixtures)
dbutils.notebook.exit(json.dumps(results))